# AnnNet benchmark

Tunable wall-time and memory profiling for the operations users hit most.

**How to use**

1. Set the parameters in the *Parameters* cell below.
2. Run the whole notebook.
3. Read the per-op tables and bar plots at the bottom.

The benchmarks are deterministic for a given seed and parameter set.


## Parameters

Edit this single cell to change graph shape / backend.


In [ ]:
# ── Parameters (edit these) ───────────────────────────────────────────
N = 5_000          # vertex count
M = 20_000         # edge count
SEED = 0
INCLUDE_HYPEREDGES = True       # also build undirected hyperedges of size 3-5
N_HYPER = 500
INCLUDE_MULTILAYER = False      # build a 2-aspect multilayer copy and benchmark supra-adj
N_LAYERS = 2

# IO benchmarks toggle (the bigger N/M, the slower)
BENCH_IO = True


## Setup


In [ ]:
import gc
import os
import time
import json
import tempfile
import warnings
from pathlib import Path

import numpy as np

import annnet
from annnet import AnnNet

try:
    import psutil
    _proc = psutil.Process(os.getpid())
    def rss_mb() -> float:
        return _proc.memory_info().rss / (1024 * 1024)
except ImportError:
    def rss_mb() -> float:
        return float('nan')

print('annnet at:', annnet.__file__)
print('starting RSS (MB):', round(rss_mb(), 2))


In [ ]:
# Simple bench harness: returns (wall_seconds, delta_rss_mb).
def bench(label, fn):
    gc.collect()
    rss0 = rss_mb()
    t0 = time.perf_counter()
    out = fn()
    elapsed = time.perf_counter() - t0
    rss1 = rss_mb()
    print(f'{label:<35} {elapsed*1000:>8.1f} ms   +{rss1 - rss0:>6.1f} MB')
    return out, elapsed, rss1 - rss0


_results: dict = {}
def record(label, elapsed, drss):
    _results[label] = {'wall_s': elapsed, 'delta_rss_mb': drss}


## Graph construction


In [ ]:
rng = np.random.default_rng(SEED)

# Vertex IDs
v_ids = [f'v{i:06d}' for i in range(N)]

# Edge endpoint pairs (sampled with replacement; no de-dup pass)
src_idx = rng.integers(0, N, size=M)
tgt_idx = rng.integers(0, N, size=M)
edge_rows = [
    {'source': v_ids[s], 'target': v_ids[t], 'edge_id': f'e{i:07d}', 'weight': float(rng.random())}
    for i, (s, t) in enumerate(zip(src_idx, tgt_idx, strict=False))
]

print(f'prepared {N:,} vertices, {M:,} edges')


In [ ]:
# Build the graph
def _build():
    G = AnnNet(directed=True)
    G.add_vertices(v_ids)
    G.add_edges_bulk(edge_rows)
    return G

G, t, d = bench('construct (add_*_bulk)', _build)
record('construct', t, d)
print('nv, ne =', G.nv, G.ne)


## Attribute bulk write


In [ ]:
# Random per-vertex attribute payload
v_attrs = {vid: {'tier': int(rng.integers(0, 4))} for vid in v_ids[:N // 2]}

_, t, d = bench('set_vertex_attrs_bulk (N/2)', lambda: G.attrs.set_vertex_attrs_bulk(v_attrs))
record('set_vertex_attrs_bulk', t, d)


In [ ]:
# Random per-edge attribute payload
e_attrs = {row['edge_id']: {'confidence': float(rng.random())} for row in edge_rows[:M // 2]}

_, t, d = bench('set_edge_attrs_bulk (M/2)', lambda: G.attrs.set_edge_attrs_bulk(e_attrs))
record('set_edge_attrs_bulk', t, d)


## Views & subgraphs


In [ ]:
_, t, d = bench('views.edges() materialize', lambda: G.views.edges())
record('views.edges', t, d)


In [ ]:
_, t, d = bench('views.vertices() materialize', lambda: G.views.vertices())
record('views.vertices', t, d)


In [ ]:
# Subgraph extraction by vertex+edge set (proxy for slice extraction)
G.slices.add('s1')
G.slices.add_edges('s1', [row['edge_id'] for row in edge_rows[: M // 4]])

s1_vertices = list(G.slices.vertices('s1'))
s1_edges = list(G.slices.edges('s1'))

_, t, d = bench(
    'ops.extract_subgraph (slice)',
    lambda: G.ops.extract_subgraph(vertices=s1_vertices, edges=s1_edges),
)
record('extract_subgraph', t, d)


## Hyperedges


In [ ]:
if INCLUDE_HYPEREDGES:
    Gh = AnnNet(directed=False)
    Gh.add_vertices(v_ids)

    members_per_he = [
        [v_ids[i] for i in rng.choice(N, size=int(rng.integers(3, 6)), replace=False)]
        for _ in range(N_HYPER)
    ]

    def _add_hypers():
        for i, members in enumerate(members_per_he):
            Gh.add_edges(members, edge_id=f'h{i:05d}')

    _, t, d = bench(f'add {N_HYPER} undirected hyperedges', _add_hypers)
    record('add_hyperedges', t, d)
    print('hyper graph nv, ne =', Gh.nv, Gh.ne)


## Multilayer supra-adjacency


In [ ]:
if INCLUDE_MULTILAYER:
    layer_labels = [f'L{i}' for i in range(N_LAYERS)]

    Gm = AnnNet(directed=False)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        Gm.layers.set_aspects(['x'], {'x': layer_labels})
        # Plant each vertex in each layer
        for L in layer_labels:
            Gm.add_vertices(v_ids[: N // N_LAYERS], layer={'x': L})
        # Intra-layer edges
        per_L = M // N_LAYERS
        for L in layer_labels:
            sub_v = v_ids[: N // N_LAYERS]
            si = rng.integers(0, len(sub_v), size=per_L)
            ti = rng.integers(0, len(sub_v), size=per_L)
            for k, (s, t) in enumerate(zip(si, ti, strict=False)):
                Gm.add_edges(
                    (sub_v[s], (L,)),
                    (sub_v[t], (L,)),
                    edge_id=f'{L}_e{k:06d}',
                    weight=1.0,
                )

    _, t, d = bench('supra_adjacency build', lambda: Gm.layers.supra_adjacency())
    record('supra_adjacency', t, d)
    _, t, d = bench('supra_laplacian build', lambda: Gm.layers.supra_laplacian())
    record('supra_laplacian', t, d)


## IO round-trips


In [ ]:
if BENCH_IO:
    tmp = Path(tempfile.mkdtemp(prefix='annnet_bench_'))
    print('temp dir:', tmp)

    from annnet import to_parquet, from_parquet
    _, t, d = bench('to_parquet', lambda: to_parquet(G, tmp / 'g_parquet'))
    record('to_parquet', t, d)
    _, t, d = bench('from_parquet', lambda: from_parquet(tmp / 'g_parquet'))
    record('from_parquet', t, d)

    from annnet import to_json, from_json
    _, t, d = bench('to_json', lambda: to_json(G, tmp / 'g.json'))
    record('to_json', t, d)
    _, t, d = bench('from_json', lambda: from_json(tmp / 'g.json'))
    record('from_json', t, d)


## Backend interop


In [ ]:
# Build the underlying nx graph (cached after first call)
_, t, d = bench('nx backend build', lambda: G.nx.backend())
record('nx_backend', t, d)


In [ ]:
# Build the underlying igraph graph
_, t, d = bench('ig backend build', lambda: G.ig.backend())
record('ig_backend', t, d)


## Results

Final wall-time and memory deltas, sorted.


In [ ]:
import polars as pl

rows = [{'op': k, 'wall_ms': v['wall_s'] * 1000, 'delta_rss_mb': v['delta_rss_mb']} for k, v in _results.items()]
df = pl.DataFrame(rows).sort('wall_ms', descending=True)
print(df)


In [ ]:
# Bar plot — wall time
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, max(3, 0.35 * len(rows))))
    plt.barh([r['op'] for r in rows], [r['wall_ms'] for r in rows])
    plt.gca().invert_yaxis()
    plt.xlabel('wall time (ms)')
    plt.title(f'annnet benchmark — N={N:,}, M={M:,}')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not available; skipping plot')


In [ ]:
# Bar plot — memory delta
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, max(3, 0.35 * len(rows))))
    plt.barh([r['op'] for r in rows], [r['delta_rss_mb'] for r in rows], color='C1')
    plt.gca().invert_yaxis()
    plt.xlabel('Δ RSS (MB)')
    plt.title(f'annnet benchmark — N={N:,}, M={M:,}')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not available; skipping plot')


In [ ]:
# Memory snapshot of the graph itself (incidence matrix + dicts + attr DFs)
graph_bytes = G.ops.memory_usage()
print(f'G.ops.memory_usage(): {graph_bytes:,} bytes  (~ {graph_bytes / (1024 * 1024):.1f} MB)')
print(f'process RSS:          {rss_mb():.1f} MB')


---

Tip: bump `N` to 50k or 200k to spot quadratic regressions early. Set
`INCLUDE_MULTILAYER=True` to add the supra-math cells to the bar plot.
